# Historical cross-region workflow / 既有跨区流程

Uses target-region normalization and the full target catalogue. Alaska no_message_attention uses the old checkpoint, not the final within-region retry. 采用目标区域尺度和完整目标目录；旧v53跨区结果不自动替换。


# Cross-region evaluation of the six controlled models

This notebook performs inference only. It loads each regional `best-model.pth` and evaluates it on all available events in the other region:

- California checkpoint -> full Alaska dataset
- Alaska checkpoint -> full California dataset

It does not train, fine-tune, update, or overwrite any model checkpoint. Metrics, per-event predictions, and four parity plots per evaluation are saved under `cross_region_results/`.

In [ ]:
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_roots = [p for p in [_start, *_start.parents] if (p / "models").is_dir() and (p / "experiments" / "paths.py").is_file()]
if not _roots:
    raise FileNotFoundError("Start Jupyter from this repository or its subdirectories.")
PACKAGE_ROOT = _roots[0]
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import gc
import os
import pickle
import random
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader
import models.transformer_attention as model_transformer_attention
import models.transformer_max as model_transformer_max
import models.no_message_attention as model_no_message_attention
import models.no_message_max as model_no_message_max
import models.gcn_max as model_gcn_max
import models.gcn_attention as model_gcn_attention

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except (AttributeError, RuntimeError):
        pass
    os.environ['PYTHONHASHSEED'] = str(seed)
SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


In [ ]:
from experiments.paths import ROOT, get_data_root, get_save_root
DATA_ROOT = get_data_root()
RESULT_ROOT = ROOT / 'cross_region_new_run'
if RESULT_ROOT.exists():
    raise FileExistsError('Choose a new output directory; existing transfer results are preserved.')
PREDICTION_DIR = RESULT_ROOT / 'predictions'
PLOT_DIR = RESULT_ROOT / 'plots'
for directory in (RESULT_ROOT, PREDICTION_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
REGIONS = {'California': {'data_dir': DATA_ROOT / 'data_DA', 'minlatitude': 32.0, 'maxlatitude': 36.0, 'minlongitude': -120.0, 'maxlongitude': -116.0, 'maxdepth': 30000.0, 'minmag': 3.0, 'maxmag': 6.0, 'km_per_degree_latitude': 110.0, 'km_per_degree_longitude': 92.0, 'plot_ranges': [(32, 36), (-120, -116), (0, 30), (3, 7)]}, 'Alaska': {'data_dir': DATA_ROOT / 'data_ANCHORAGE_DA', 'minlatitude': 59.0, 'maxlatitude': 63.0, 'minlongitude': -153.0, 'maxlongitude': -147.0, 'maxdepth': 250000.0, 'minmag': 3.0, 'maxmag': 6.0, 'km_per_degree_latitude': 111.0, 'km_per_degree_longitude': 54.0, 'plot_ranges': [(59, 63), (-153, -147), (0, 250), (3, 7)]}}
MODEL_SPECS = [{'key': 'transformer_attention', 'name': 'GraphTransformer_AttentionPooling', 'family': 'graph', 'module': model_transformer_attention}, {'key': 'transformer_max', 'name': 'GraphTransformer_MaxPooling', 'family': 'graph', 'module': model_transformer_max}, {'key': 'no_message_attention', 'name': 'NoMessagePassing_AttentionPooling', 'family': 'graph', 'module': model_no_message_attention}, {'key': 'no_message_max', 'name': 'NoMessagePassing_MaxPooling', 'family': 'graph', 'module': model_no_message_max}, {'key': 'gcn_max', 'name': 'GCN_MaxPooling', 'family': 'graph', 'module': model_gcn_max}, {'key': 'gcn_attention', 'name': 'GCN_AttentionPooling', 'family': 'graph', 'module': model_gcn_attention}]

def checkpoint_path(spec, source_region):
    import json, hashlib
    table = json.loads((ROOT / 'experiments' / 'weights.json').read_text(encoding='utf-8'))['cross_region_original']
    row = table[source_region + '/' + spec['key']]
    path = get_save_root().joinpath(*Path(row['relative_path']).parts[1:])
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != row['sha256']:
        raise ValueError('Checkpoint hash differs from the archived cross-region experiment: ' + str(path))
    return path
checks = []
for source_region in REGIONS:
    for spec in MODEL_SPECS:
        path = checkpoint_path(spec, source_region)
        checks.append({'model': spec['key'], 'source_region': source_region, 'checkpoint_exists': path.is_file(), 'checkpoint': str(path)})
checkpoint_table = pd.DataFrame(checks)
display(checkpoint_table)
missing = checkpoint_table.loc[~checkpoint_table['checkpoint_exists']]
if not missing.empty:
    raise FileNotFoundError('Some checkpoints are missing:\n' + '\n'.join(missing['checkpoint']))
for region_name, cfg in REGIONS.items():
    required = [cfg['data_dir'] / 'catalogue.csv', cfg['data_dir'] / 'stations.csv', cfg['data_dir'] / 'catalogue_station_lookup_final.pickle', cfg['data_dir'] / 'waveforms_proc_broadband']
    for path in required:
        if not path.exists():
            raise FileNotFoundError(f'Missing {region_name} data path: {path}')
print(f'Release root: {ROOT}')
print(f'Data root:    {DATA_ROOT}')
print(f'Results:      {RESULT_ROOT}')


## 数据协议 / Data protocol

采用目标地区边界归一化和反归一化，并使用目标地区完整可用事件集。
Target-region bounds and the full available target catalogue are used; this is not the within-region test subset.


In [ ]:
N_SUB = 50
N_T = 2048
BATCH_SIZE = 32

def prepare_target_data(region_name, family):
    cfg = REGIONS[region_name]
    data_dir = cfg['data_dir']
    waveform_dir = data_dir / 'waveforms_proc_broadband'
    catalogue_frame = pd.read_csv(data_dir / 'catalogue.csv')[['lat', 'lon', 'depth', 'mag']]
    has_waveform = np.array([(waveform_dir / f'{int(event_id)}.npy').is_file() for event_id in catalogue_frame.index])
    catalogue_frame = catalogue_frame.loc[has_waveform].copy()
    with (data_dir / 'catalogue_station_lookup_final.pickle').open('rb') as stream:
        lookup = pickle.load(stream)
    event_ids = catalogue_frame.index.to_numpy(dtype=np.int64)
    values = catalogue_frame.to_numpy(dtype=np.float64)
    values[:, 0] = (values[:, 0] - cfg['minlatitude']) / (cfg['maxlatitude'] - cfg['minlatitude'])
    values[:, 1] = (values[:, 1] - cfg['minlongitude']) / (cfg['maxlongitude'] - cfg['minlongitude'])
    values[:, 2] = values[:, 2] / cfg['maxdepth']
    values[:, 3] = (values[:, 3] - cfg['minmag']) / (cfg['maxmag'] - cfg['minmag'])
    values = (values - 0.5) * 2.0
    weights = np.ones((len(event_ids), 1), dtype=np.float64)
    catalogue = np.concatenate([event_ids.reshape(-1, 1), weights, values], axis=1)
    stations = pd.read_csv(data_dir / 'stations.csv')[['code', 'lat', 'lon']]
    stations['lat'] = ((stations['lat'] - cfg['minlatitude']) / (cfg['maxlatitude'] - cfg['minlatitude']) - 0.5) * 2.0
    stations['lon'] = ((stations['lon'] - cfg['minlongitude']) / (cfg['maxlongitude'] - cfg['minlongitude']) - 0.5) * 2.0
    set_seed(SEED)
    dataset_class = model_transformer_attention.SeismicDataset1
    dataset = dataset_class(data_dir=str(waveform_dir), catalogue=catalogue, stations=stations, lookup=lookup, N_sub=N_SUB, N_t=N_T)
    ordered_event_ids = catalogue[dataset.event_inds.astype(int), 0].astype(np.int64)
    if len(ordered_event_ids) != len(dataset):
        raise RuntimeError('Event identifiers are not aligned with the generated dataset.')
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f'{region_name:10s} | {family:6s} | test events: {len(dataset)}')
    return (dataset, loader, ordered_event_ids)


In [ ]:
def build_model(spec):
    set_seed(SEED)
    model = spec['module'].GraphNet(activation='relu')
    return model.to(DEVICE)

def load_best_model(spec, source_region):
    path = checkpoint_path(spec, source_region)
    model = build_model(spec)
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return (model, path)

def collect_predictions(model, loader, family):
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels, _ in loader:
            labels = labels.to(DEVICE)
            waveforms, coords, weights = inputs
            predictions = model(waveforms.to(DEVICE), coords.to(DEVICE), weights.to(DEVICE))
            all_predictions.append(predictions.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    return (np.vstack(all_predictions), np.vstack(all_labels))

def unscale(values, region_name, family):
    cfg = REGIONS[region_name]
    values = np.asarray(values, dtype=np.float64)
    values = values / 2.0 + 0.5
    latitude = values[:, 0] * (cfg['maxlatitude'] - cfg['minlatitude']) + cfg['minlatitude']
    longitude = values[:, 1] * (cfg['maxlongitude'] - cfg['minlongitude']) + cfg['minlongitude']
    depth_km = values[:, 2] * cfg['maxdepth'] / 1000.0
    magnitude = values[:, 3] * (cfg['maxmag'] - cfg['minmag']) + cfg['minmag']
    return np.stack([latitude, longitude, depth_km, magnitude], axis=1)


In [ ]:
VARIABLES = ['Latitude', 'Longitude', 'Depth_km', 'Magnitude']

def save_predictions(spec, source_region, target_region, event_ids, true_values, predictions):
    output_path = PREDICTION_DIR / f"{spec['key']}_{source_region}_to_{target_region}_predictions.csv"
    frame = pd.DataFrame({'event_id': event_ids, 'true_latitude_deg': true_values[:, 0], 'predicted_latitude_deg': predictions[:, 0], 'true_longitude_deg': true_values[:, 1], 'predicted_longitude_deg': predictions[:, 1], 'true_depth_km': true_values[:, 2], 'predicted_depth_km': predictions[:, 2], 'true_magnitude': true_values[:, 3], 'predicted_magnitude': predictions[:, 3]})
    frame.to_csv(output_path, index=False, float_format='%.17g')
    return output_path

def metric_rows(spec, source_region, target_region, checkpoint, prediction_path, true_values, predictions):
    cfg = REGIONS[target_region]
    rows = []
    for index, variable in enumerate(VARIABLES):
        mae = mean_absolute_error(true_values[:, index], predictions[:, index])
        mse = mean_squared_error(true_values[:, index], predictions[:, index])
        r2 = r2_score(true_values[:, index], predictions[:, index])
        if index == 0:
            km_scale = cfg['km_per_degree_latitude']
        elif index == 1:
            km_scale = cfg['km_per_degree_longitude']
        else:
            km_scale = np.nan
        rows.append({'model': spec['key'], 'model_name': spec['name'], 'source_region': source_region, 'target_region': target_region, 'test_events': len(true_values), 'variable': variable, 'MAE_native': mae, 'MSE_native': mse, 'R2': r2, 'MAE_km': mae * km_scale if np.isfinite(km_scale) else np.nan, 'MSE_km2': mse * km_scale ** 2 if np.isfinite(km_scale) else np.nan, 'checkpoint': str(checkpoint.resolve()), 'prediction_file': str(prediction_path.resolve())})
    return rows

def save_parity_plots(spec, source_region, target_region, true_values, predictions):
    ranges = REGIONS[target_region]['plot_ranges']
    plot_names = ['Latitude', 'Longitude', 'Depth_km', 'Magnitude']
    for index, name in enumerate(plot_names):
        x_true = true_values[:, index]
        y_pred = predictions[:, index]
        mae = np.mean(np.abs(x_true - y_pred))
        mse = np.mean((x_true - y_pred) ** 2)
        r2 = r2_score(x_true, y_pred)
        figure, axis = plt.subplots(figsize=(5, 5))
        axis.scatter(x_true, y_pred, s=30, alpha=0.5, c='blue', marker='o', edgecolors='none')
        center_min, center_max = ranges[index]
        pad = 0.05 * (center_max - center_min)
        axis.plot([center_min, center_max], [center_min, center_max], 'r--', linewidth=1)
        axis.set_xlim(center_min - pad, center_max + pad)
        axis.set_ylim(center_min - pad, center_max + pad)
        axis.set_xlabel(f"True {name.replace('_', ' ')}")
        axis.set_ylabel(f"Predicted {name.replace('_', ' ')}")
        metrics_text = f'MAE = {mae:.4f}\nMSE = {mse:.4f}\n$R^2$ = {r2:.4f}'
        axis.text(0.05, 0.95, metrics_text, transform=axis.transAxes, ha='left', va='top', bbox=dict(boxstyle='round', alpha=0.2, pad=0.3))
        axis.grid(True, linestyle='--', alpha=0.5)
        axis.set_aspect('equal', adjustable='box')
        if index != 2:
            axis.xaxis.set_major_locator(mticker.MultipleLocator(1))
            axis.yaxis.set_major_locator(mticker.MultipleLocator(1))
        figure.tight_layout()
        output_path = PLOT_DIR / f"{spec['key']}_{source_region}_to_{target_region}_{index:02d}_{name}.png"
        figure.savefig(output_path, dpi=600)
        plt.close(figure)


## Run all 12 cross-region evaluations

Results are written after every model, so completed evaluations remain available if execution is interrupted. Dataset objects are released between model families to limit memory use.

In [ ]:
all_metrics = []
metrics_path = RESULT_ROOT / 'metrics.csv'
families = [('graph', [spec for spec in MODEL_SPECS if spec['family'] == 'graph'])]
for target_region in ('Alaska', 'California'):
    source_region = 'California' if target_region == 'Alaska' else 'Alaska'
    print('\n' + '=' * 78)
    print(f'{source_region} checkpoints -> full {target_region} dataset')
    print('=' * 78)
    for family, specs in families:
        dataset, loader, event_ids = prepare_target_data(target_region, family)
        for spec in specs:
            print(f"\nEvaluating {spec['key']} ({spec['name']})")
            model, checkpoint = load_best_model(spec, source_region)
            normalized_predictions, normalized_labels = collect_predictions(model, loader, family)
            predictions = unscale(normalized_predictions, target_region, family)
            true_values = unscale(normalized_labels, target_region, family)
            prediction_path = save_predictions(spec, source_region, target_region, event_ids, true_values, predictions)
            all_metrics.extend(metric_rows(spec, source_region, target_region, checkpoint, prediction_path, true_values, predictions))
            pd.DataFrame(all_metrics).to_csv(metrics_path, index=False, float_format='%.17g')
            save_parity_plots(spec, source_region, target_region, true_values, predictions)
            print(f'Saved predictions: {prediction_path.name}')
            print(f'Updated metrics:   {metrics_path}')
            del model, normalized_predictions, normalized_labels, predictions, true_values
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        del loader, dataset, event_ids
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print('\nAll cross-region evaluations completed.')


In [ ]:
metrics = pd.read_csv(RESULT_ROOT / 'metrics.csv')
display(metrics)
summary = metrics.pivot_table(index=['source_region', 'target_region', 'model', 'model_name'], columns='variable', values=['MAE_native', 'MAE_km', 'MSE_native', 'R2'], aggfunc='first')
display(summary)
print(f"Metrics:     {RESULT_ROOT / 'metrics.csv'}")
print(f'Predictions: {PREDICTION_DIR}')
print(f'Plots:       {PLOT_DIR}')
